<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/main/notebooks/01_Data_Creation_QP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import os
from google.colab import drive
from pathlib import Path
import shutil
import json

In [ ]:
print(requests.__version__)

2.32.4


### Binance API

In [ ]:
# Binance API address

base_url = "https://data-api.binance.vision"
endpoint = "/api/v3/klines"

url = base_url + endpoint

print(url)

https://data-api.binance.vision/api/v3/klines


### Trial Data Pull

In [ ]:
# request parameters - creation

params = {
    "symbol": "BTCUSDT",
    "interval": "1d",
    "limit": 5
}

In [ ]:
# getting the response for the sent parameters

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

In [ ]:
# check if the trial request succeeded

# 200 = request succeeded
# 400 = incorrect request
# 403 = access forbidden
# 404 = endpoint not found
# 429 = too many requests
# 500 = server-side problem

print(response.status_code)

200


In [ ]:
# Data Process to required format

data = response.json()

print(data)
print(len(data))

# | Position | Meaning                  |
# | -------: | ------------------------ |
# |        0 | Candle opening timestamp |
# |        1 | Open price               |
# |        2 | Highest price            |
# |        3 | Lowest price             |
# |        4 | Close price              |
# |        5 | Base-asset volume        |
# |        6 | Candle closing timestamp |
# |        7 | Quote-asset volume       |
# |        8 | Number of trades         |
# |        9 | Taker-buy base volume    |
# |       10 | Taker-buy quote volume   |
# |       11 | Unused field             |


[[1784246400000, '63830.20000000', '64387.99000000', '62537.56000000', '63931.67000000', '18189.89389000', 1784332799999, '1152546029.34025150', 3239296, '8618.43801000', '545926712.67857150', '0'], [1784332800000, '63931.67000000', '64865.00000000', '63886.65000000', '64834.22000000', '8031.18012000', 1784419199999, '515954648.68636990', 1088701, '4100.21428000', '263412780.44647870', '0'], [1784419200000, '64834.21000000', '64967.25000000', '64280.00000000', '64722.54000000', '8379.91474000', 1784505599999, '541559101.91299460', 1464337, '4281.97001000', '276728781.26507170', '0'], [1784505600000, '64722.55000000', '65799.00000000', '63100.00000000', '65255.51000000', '21323.48702000', 1784591999999, '1379182710.26304440', 3606847, '10104.82668000', '654547128.90542850', '0'], [1784592000000, '65255.51000000', '66956.15000000', '65148.75000000', '66559.99000000', '14739.43187000', 1784678399999, '975426778.62292520', 2297303, '7720.69061000', '510915702.70660320', '0']]
5


### Structured - Paginated data pull

In [ ]:
# converting to pandas DF

columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "close_time",
    "quote_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

numeric_columns = [
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "quote_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume"
]

# btc_df = pd.DataFrame(
#     data,
#     columns=columns
# )

In [ ]:
def data_process(df, numeric_columns, asset, symbol):

  df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
  df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

  df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric)
  df = df.drop(columns=["ignore"])

  df["asset"] = asset
  df["symbol"] = symbol

  df = df[
    [
        "open_time",
        "close_time",
        "asset",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "base_volume",
        "quote_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume"
    ]
  ]

  return df

In [ ]:
# return & Volatility Calculations

def return_volatility_calc(df):

    df = (
        df.sort_values("open_time")
          .drop_duplicates(subset=["open_time"])
          .reset_index(drop=True)
    )

    df["close_return"] = (
        df["close"] / df["close"].shift(1) - 1
    )

    df["intraday_return"] = (
        df["close"] / df["open"] - 1
    )

    df["intraday_range"] = (
        (df["high"] - df["low"]) / df["open"]
    )

    df["base_volume_return"] = (
        df["base_volume"].pct_change(fill_method=None)
    )

    df["quote_volume_return"] = (
        df["quote_volume"].pct_change(fill_method=None)
    )

    df["log_return"] = np.log(
        df["close"] / df["close"].shift(1)
    )

    return df

In [ ]:
def interval_to_milliseconds(interval):

    unit = interval[-1]
    value = int(interval[:-1])

    if unit == "m":
        return value * 60 * 1000

    elif unit == "h":
        return value * 60 * 60 * 1000

    elif unit == "d":
        return value * 24 * 60 * 60 * 1000

    elif unit == "w":
        return value * 7 * 24 * 60 * 60 * 1000

    else:
        raise ValueError(
            f"Unsupported interval: {interval}"
        )

In [ ]:
def download_raw_klines(symbol, interval, start_date, end_date, limit=1000):

    start_timestamp = pd.Timestamp(
        start_date,
        tz="UTC"
    )

    end_timestamp = pd.Timestamp(
        end_date,
        tz="UTC"
    )

    current_start_ms = int(
        start_timestamp.timestamp() * 1000
    )

    end_time_ms = int(
        end_timestamp.timestamp() * 1000
    )

    interval_ms = interval_to_milliseconds(
        interval
    )

    all_data = []

    while current_start_ms < end_time_ms:

        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start_ms,
            "endTime": end_time_ms,
            "limit": limit
        }

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        batch = response.json()

        if len(batch) == 0:
            break

        all_data.extend(batch)

        last_open_time = batch[-1][0]

        current_start_ms = (
            last_open_time + interval_ms
        )

        print(
            f"Downloaded {len(all_data)} rows "
            f"for {symbol}"
        )

        time.sleep(0.2)

    return all_data

In [ ]:
 # Getting all the relevant data togethor

def get_crypto_data(symbol, asset, columns, numeric_columns, start_date, end_date, interval="1d", limit=1000):

    raw_data = download_raw_klines(symbol, interval, start_date, end_date, limit=limit)

    df = pd.DataFrame(
        raw_data,
        columns=columns
    )

    df = (
    df.sort_values("open_time")
      .drop_duplicates(subset=["open_time"])
      .reset_index(drop=True)
    )

    df = data_process(df, numeric_columns, asset, symbol)
    df = return_volatility_calc(df)

    return df


In [ ]:
# Checking the function

assets = {
    "BTC": "BTCUSDT",
    "ETH": "ETHUSDT",
    "SOL": "SOLUSDT",
    "XRP": "XRPUSDT"
}

start_date = "2016-07-01"
end_date = "2026-07-01"

crypto_dfs = []

for asset, symbol in assets.items():
  crypto_dfs.append(get_crypto_data(symbol, asset, columns, numeric_columns, start_date, end_date, interval="15m", limit=1000))


crypto_df = pd.concat(
        crypto_dfs,
    ignore_index=True
)

crypto_df = (
    crypto_df
    .sort_values(
        ["open_time", "asset"]
    )
    .reset_index(drop=True)
)

### Saving and Sharing the DF

In [ ]:
temporary_path = "/content/crypto_15m_data.parquet"

crypto_df.to_parquet(
    temporary_path,
    index=False
)

In [ ]:

file_size_mb = os.path.getsize(temporary_path) / (1024 ** 2)

print(f"File size: {file_size_mb:.2f} MB")

File size: 117.45 MB


In [ ]:
test_df = pd.read_parquet(temporary_path)

print(test_df.shape)
print(test_df.dtypes)
assert len(test_df) == len(crypto_df)

(1112863, 19)
open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
close_return                          float64
intraday_return                       float64
intraday_range                        float64
base_volume_return                    float64
quote_volume_return                   float64
log_return                            float64
dtype: object


In [ ]:
# Accessing the google drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
shared_project_dir = Path(
    "/content/drive/MyDrive/Quant Research"
)

print(shared_project_dir.exists())

True


In [ ]:

data_dir = shared_project_dir / "data"
processed_dir = data_dir / "processed"

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(processed_dir)

# interval = "15m"
# dataset_start = start_date
# dataset_end = end_date

# filename = (
#     f"crypto_binance_{interval}_"
#     f"{dataset_start}_{dataset_end}_v1.parquet"
# )

filename = "crypto_binance_15m_btc_eth_sol_xrp_v1.parquet"

shared_file_path = (
    processed_dir /
    filename
)

shutil.copy2(
    temporary_path,
    shared_file_path
)

print("Saved to:")
print(shared_file_path)

/content/drive/MyDrive/Quant Research/data/processed
Saved to:
/content/drive/MyDrive/Quant Research/data/processed/crypto_binance_15m_btc_eth_sol_xrp_v1.parquet


In [ ]:
# checking if export was correct

drive_df = pd.read_parquet(shared_file_path)

print(drive_df.shape)

assert len(drive_df) == len(crypto_df)

(1112863, 19)


In [ ]:
print(shared_file_path.exists())
print(
    f"{shared_file_path.stat().st_size / 1024**2:.2f} MB"
)

True
117.45 MB


### Manifest Creation

In [ ]:
summary = (
    crypto_df.groupby("asset")
    .agg(
        rows=("open_time", "size"),
        first_timestamp=("open_time", "min"),
        last_timestamp=("open_time", "max")
    )
    .reset_index()
)

summary["first_timestamp"] = (
    summary["first_timestamp"].astype(str)
)

summary["last_timestamp"] = (
    summary["last_timestamp"].astype(str)
)

manifest = {
    "dataset_name": filename,
    "source": "Binance Spot API",
    "interval": "15m",
    "assets": list(assets.keys()),
    "symbols": assets,
    "total_rows": int(len(crypto_df)),
    "columns": crypto_df.columns.tolist(),
    "asset_summary": summary.to_dict(
        orient="records"
    )
}

In [ ]:
manifest_path = (
    shared_project_dir / "manifests" /
    "crypto_binance_15m_btc_eth_sol_xrp_manifest_v1.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2
    )

print(manifest_path)

/content/drive/MyDrive/Quant Research/manifests/crypto_binance_15m_btc_eth_sol_xrp_manifest_v1.json


### Other Data Checks

In [ ]:
crypto_df.groupby("asset").size()

,0
asset,
BTC,310460
ETH,310460
SOL,206283
XRP,285660


In [ ]:
crypto_df.groupby("asset").agg(
    first_date=("open_time", "min"),
    last_date=("open_time", "max"),
    rows=("open_time", "size")
)

,first_date,last_date,rows
asset,,,
BTC,2017-08-17 04:00:00+00:00,2026-07-01 00:00:00+00:00,310460
ETH,2017-08-17 04:00:00+00:00,2026-07-01 00:00:00+00:00,310460
SOL,2020-08-11 06:00:00+00:00,2026-07-01 00:00:00+00:00,206283
XRP,2018-05-04 08:00:00+00:00,2026-07-01 00:00:00+00:00,285660


In [ ]:
print(crypto_df.shape)
print()
print(crypto_df.dtypes)

(1112863, 19)

open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
close_return_dod                      float64
intraday_return_pct                   float64
intraday_range_pct                    float64
base_volume_dod_pct                   float64
quote_volume_dod_pct                  float64
log_return                            float64
dtype: object


In [ ]:
crypto_df.duplicated(
    subset=["asset", "open_time"]
).sum()

np.int64(0)

In [ ]:
crypto_df.isna().sum()

,0
open_time,0
close_time,0
asset,0
symbol,0
open,0
high,0
low,0
close,0
base_volume,0
quote_volume,0


In [ ]:
crypto_df[crypto_df.asset == 'SOL'].open_time.min()

Timestamp('2020-08-11 00:00:00+0000', tz='UTC')

In [ ]:
crypto_df.groupby("asset").size()

,0
asset,
BTC,5
ETH,5
SOL,5


In [ ]:
print(crypto_df.shape)
print()
print(crypto_df.dtypes)

(124, 19)

open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
close_return_dod                      float64
intraday_return_pct                   float64
intraday_range_pct                    float64
base_volume_dod_pct                   float64
quote_volume_dod_pct                  float64
log_return                            float64
dtype: object


In [ ]:
btc_df = data_process(btc_df, numeric_columns, 'BTC', 'BTCUSDT')
btc_df = return_volatility_calc(btc_df)

print(btc_df.shape)
print()
print(btc_df.dtypes)

(5, 19)

open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
close_return_dod                      float64
intraday_return_pct                   float64
intraday_range_pct                    float64
base_volume_dod_pct                   float64
quote_volume_dod_pct                  float64
log_return                            float64
dtype: object


In [ ]:
btc_df

,open_time,close_time,asset,symbol,open,high,low,close,base_volume,quote_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,close_return_dod,intraday_return_pct,intraday_range_pct,base_volume_dod_pct,quote_volume_dod_pct,log_return
0,2026-07-17 00:00:00+00:00,2026-07-17 23:59:59.999000+00:00,BTC,BTCUSDT,63830.20,64387.99,62537.56,63931.67,18189.89389,1.152546e+09,3239296,8618.43801,5.459267e+08,NaN,0.001590,0.028990,NaN,NaN,NaN
1,2026-07-18 00:00:00+00:00,2026-07-18 23:59:59.999000+00:00,BTC,BTCUSDT,63931.67,64865.00,63886.65,64834.22,8031.18012,5.159546e+08,1088701,4100.21428,2.634128e+08,0.014117,0.014117,0.015303,-0.558481,-0.552335,0.014019
2,2026-07-19 00:00:00+00:00,2026-07-19 23:59:59.999000+00:00,BTC,BTCUSDT,64834.21,64967.25,64280.00,64722.54,8379.91474,5.415591e+08,1464337,4281.97001,2.767288e+08,-0.001723,-0.001722,0.010600,0.043423,0.049625,-0.001724
3,2026-07-20 00:00:00+00:00,2026-07-20 23:59:59.999000+00:00,BTC,BTCUSDT,64722.55,65799.00,63100.00,65255.51,21323.48702,1.379183e+09,3606847,10104.82668,6.545471e+08,0.008235,0.008235,0.041701,1.544595,1.546689,0.008201
4,2026-07-21 00:00:00+00:00,2026-07-21 23:59:59.999000+00:00,BTC,BTCUSDT,65255.51,66354.00,65148.75,66323.99,6483.59829,4.261850e+08,1028206,3414.65942,2.244275e+08,0.016374,0.016374,0.018470,-0.695941,-0.690987,0.016241
